# Photo to 3D model, with real measurements

Upload a photo of an object with a printed marker or a bank card beside it. Get back a
spinnable 3D model plus its length, width, height and volume in millimetres.

**Why this notebook is shaped the way it is.** Colab ships numpy, Pillow, scipy, OpenCV and
torch already compiled against each other. Every separate `pip install` is a separate
resolution, free to move one of them to satisfy some new package's pin, and a partly applied
downgrade leaves a package's Python files and its compiled extension on different versions.
The error you get then names the wrong culprit: a broken numpy reports itself as "rembg is
not installed". Chasing those one at a time never converges. So:

1. **One install, one transaction** (step 3). It pins the versions this machine already has
   and installs everything else inside those bounds.
2. **One restart**, immediately after (step 4). The kernel is holding the old modules.
3. **Verify before spending GPU time** (step 5). The doctor runs the whole measurement
   pipeline on a synthetic scene of known size, in a fresh process.

Run the steps in order. If one fails, it says which layer failed and what to do.

## 1. Which GPU did we get?

Runtime > Change runtime type > T4 GPU. A T4 is enough: shape generation needs about 10 GB.
You can run everything except generation with no GPU at all.

In [ ]:
!nvidia-smi || echo "No GPU. Set Runtime > Change runtime type > T4 GPU."

## 2. Session setup: Drive cache, code, working directory

Safe to re-run, and you **will** re-run it after the restart in step 4 - a restart clears
environment variables and drops you back in `/content`.

Drive holds two caches: model weights, which are gigabytes and would otherwise download every
session, and the pip cache, which makes later installs much faster. Set `USE_DRIVE = False`
to skip both.

In [ ]:
USE_DRIVE = True
CODE_SOURCE = "github"  # "github" or "drive"
GITHUB_REPO = "https://github.com/Dnyaneshwarigund12/3d_model_generation.git"
GITHUB_BRANCH = "main"
DRIVE_PROJECT = "/content/drive/MyDrive/3d_model_generation"
PROJECT_DIR = "/content/3d_model_generation"

import os
import shutil
import subprocess

if USE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    cache = "/content/drive/MyDrive/p3d-cache"
    os.makedirs(f"{cache}/pip", exist_ok=True)
    os.environ["HF_HOME"] = f"{cache}/huggingface"  # model weights
    os.environ["TORCH_HOME"] = f"{cache}/torch"
    os.environ["U2NET_HOME"] = f"{cache}/rembg"  # background removal weights
    os.environ["PIP_CACHE_DIR"] = f"{cache}/pip"  # wheels, so step 3 is fast next time
    print("caching weights and wheels in", cache)
else:
    print("not using Drive: weights and wheels download again every session")

if CODE_SOURCE == "github":
    if os.path.isdir(PROJECT_DIR):
        subprocess.run(["git", "-C", PROJECT_DIR, "pull", "--ff-only"], check=False)
    else:
        subprocess.run(
            ["git", "clone", "--branch", GITHUB_BRANCH, GITHUB_REPO, PROJECT_DIR],
            check=True,
        )
elif CODE_SOURCE == "drive":
    if not os.path.isdir(DRIVE_PROJECT):
        raise SystemExit(f"{DRIVE_PROJECT} not found. Upload the project folder to Drive.")
    if os.path.isdir(PROJECT_DIR):
        shutil.rmtree(PROJECT_DIR)
    shutil.copytree(DRIVE_PROJECT, PROJECT_DIR)
else:
    raise SystemExit("CODE_SOURCE must be 'github' or 'drive'.")

os.chdir(PROJECT_DIR)
print("working in", os.getcwd())

## 3. Install everything, once

`tools/colab_setup.py` writes `constraints-colab.txt` from the versions this machine already
has - numpy, Pillow, scipy, OpenCV, numba, torch - and then installs the pipeline and both
generation backends in a single pip transaction bounded by that file. Nothing can move the
baseline underneath the rest of the stack. If a package genuinely cannot live with Colab's
numpy or torch, pip says so here, by name, instead of leaving you a broken import later.

Five to ten minutes the first time; much less afterwards with the pip cache on Drive. Pass
`--backend triposr` or `--backend hunyuan3d` to prepare only one - the default does both,
because a second transaction later is exactly what breaks the first.

It ends by telling you whether anything moved. Model weights are not downloaded here; they
arrive on first use.

In [ ]:
!python tools/colab_setup.py

## 4. Restart the runtime now

**Runtime > Restart session**, or Ctrl+M then full stop.

This is not superstition. This kernel imported whatever was on disk when step 2 ran, and
Python cannot reload a compiled extension in place, so anything step 3 changed is invisible
until the process restarts.

Afterwards, re-run **step 2** - it restores the cache paths and the working directory - and
carry on from step 5. **Do not re-run step 3.** Nothing is missing, and a second install is
the one thing that can undo the first.

## 5. Doctor: is everything actually working?

This runs as a script, in a fresh process, because that is the only place a version number
reflects what is on disk rather than what a kernel imported first.

It checks each library, then runs the real pipeline end to end on a synthetic scene: a
300x150 px object beside a 150 px marker that is really 50 mm, which has to come back as
100 x 50 mm. That single check covers marker detection, the millimetres-per-pixel maths, mesh
scaling, the oriented bounding box and the GLB export.

WARN on the GPU or a backend is fine if you do not need it. Any FAIL means stop here: re-run
step 3, restart, and run this again. Add `--verbose` for full tracebacks.

In [ ]:
!python tools/doctor.py

## 6. Print a marker

This is what makes the measurements real rather than a guess. Download the PNG, print it at
**100% scale** (turn off "fit to page"), measure the printed black square with a ruler, and
type the measured value into the app.

Lay it flat beside the object, roughly in the same plane, with all four corners visible.

In [ ]:
!python tools/make_marker.py --mm 50 --pdf --out assets/markers/marker_50mm.png

from IPython.display import Image, display

display(Image("assets/markers/marker_50mm.png", width=320))

from google.colab import files

files.download("assets/markers/marker_50mm.pdf")

## 7. Launch the app

`share=True` gives a public `gradio.live` URL, valid for 72 hours and only while this cell
keeps running. Open it on your phone to upload photos directly.

Weights load on the first request, so the first model takes noticeably longer than the rest.

Start on `triposr`: a second or two per image, lowest quality, enough to confirm the flow.
Then switch to `hunyuan3d` for the quality you would actually ship - 35 to 50 seconds per
image on a T4 in low-VRAM mode. `silhouette` is a CPU placeholder for testing without a GPU.

In [ ]:
GENERATOR = "triposr"  # "triposr", "hunyuan3d", or "silhouette"

from app.config import Settings
from app.ui import build_ui

settings = Settings.from_env()
settings.generator = GENERATOR
settings.low_vram = True
settings.hunyuan_texture = False  # ~21 GB on its own; too much for a T4

build_ui(settings).queue().launch(share=True, show_error=True)

## 8. Or run it headless on one image

Useful for debugging: it prints every stage's timing and the full result, and writes
everything to `outputs/<run_id>/`.

In [ ]:
import json

from google.colab import files

from app.config import Settings
from app.pipeline import run

uploaded = files.upload()
photo = next(iter(uploaded))

settings = Settings.from_env()
settings.generator = "triposr"

result = run(photo, settings=settings, scale_source="marker", marker_mm=50.0)

print(result.summary)
print("timings:", result.timings_s)
for warning in result.warnings:
    print("warning:", warning)
print(json.dumps(result.measurements, indent=2))
print("outputs in", result.run_dir)

## 9. Are the numbers actually right?

The error percentages the app reports start as published estimates, not measurements of this
pipeline. To replace them with real ones: tape-measure 10-15 objects, photograph each with the
marker, fill in a CSV like `tools/validation_manifest.example.csv`, and run the cell below.
It prints measured error per tier.

In [ ]:
!python tools/validate.py --manifest tools/my_objects.csv --generator triposr

## If a library import breaks anyway

Symptom: an ImportError from inside numpy or Pillow itself, such as `cannot import name
'_center' from 'numpy._core.umath'`. Something installed outside step 3 moved a package, and
the move was applied only partly, so its Python files and its compiled extension disagree.

Installing over the top may not fix it: pip records the new version and can leave stale files
in place, so the metadata claims one version while the files are another. Delete the tree and
rebuild it, then restart the runtime.

In [ ]:
PACKAGE, IMPORT_AS = "numpy", "numpy"  # or "pillow", "PIL"

import glob
import shutil
import subprocess
import sys
import sysconfig

site = sysconfig.get_paths()["purelib"]
wanted = dict(
    line.split("==") for line in open("constraints-colab.txt") if "==" in line
)
pin = wanted[PACKAGE].strip()

subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", PACKAGE], check=False)
for name in {PACKAGE, PACKAGE.capitalize(), IMPORT_AS}:
    for leftover in (
        glob.glob(f"{site}/{name}")
        + glob.glob(f"{site}/{name}-*.dist-info")
        + glob.glob(f"{site}/{name}.libs")
    ):
        print("removing", leftover)
        shutil.rmtree(leftover, ignore_errors=True)

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir", f"{PACKAGE}=={pin}"],
    check=True,
)

# Verify in a separate process: this kernel still holds the broken copy.
probe = subprocess.run(
    [sys.executable, "-c", f"import {IMPORT_AS}; print({IMPORT_AS}.__version__)"],
    capture_output=True,
    text=True,
)
print("a fresh process now sees:", probe.stdout.strip() or probe.stderr.strip())
print("\nRESTART THE RUNTIME, re-run step 2, then step 5.")